# Reltio & Fleet Segment Attrition Audit Notebook (v3)

This notebook audits and compares our attrition model's segmentation and metrics against the Reltio-based analysis from the other team. 

### Why we start from the pipeline tables (`C2` and `C3`):
By querying the intermediate tables produced during our pipeline execution (`WORKSPACE.digitalda_stage.Account_ID_features_monthly` for C2, and the SCD2 Reltio bridge from C3), we guarantee:
1. **100% data parity** with what the model actually uses.
2. **Pre-filtered and clean metrics** (e.g., correct FICO, partner indicator, and account number lengths/types).
3. **Consistent mapping logic** for retroactive organization history before March 2026.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Append 'py' directory to path to import data_loader
sys.path.append('py')
from data_loader import get_snowpark_session

print("Libraries loaded successfully.")

## 1. Connect to Snowflake and Load Pipeline Data

We query the C2 account monthly table and map it to organizations using the C3 bridge definition:

In [ ]:
session = get_snowpark_session()

query = """
WITH 
RELTIO_BRIDGE AS (
    -- Replicate C3 SCD2 bridge to map account_id to org_uri for each month
    SELECT DISTINCT
        wx.accountnumber AS ACCOUNT_ID,
        hb.org_uri AS ORG_URI,
        hb.cohort_month AS COHORT_MONTH
    FROM WORKSPACE.digitalda_stage.Account_ID_features_monthly af
    JOIN PREP.MDM_RELTIO.entity_wxaccountnumber wx ON af.account_id = wx.accountnumber
    JOIN (
        WITH anchor_records AS (
            SELECT 
                account_uri,
                organization_uri AS org_uri
            FROM PREP.MDM_RELTIO.f_entity_wxaccountnumber_organization_snapshot
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY account_uri 
                ORDER BY
                    CASE WHEN row_eff_begin_dttm <= '2026-03-01'::DATE AND (row_eff_end_dttm IS NULL OR row_eff_end_dttm > '2026-03-01'::DATE) THEN 1
                    WHEN row_eff_begin_dttm > '2026-03-01'::DATE THEN 2
                    ELSE 3 END,
                CASE WHEN row_eff_begin_dttm > '2026-03-01'::DATE THEN row_eff_begin_dttm END ASC,
                row_eff_end_dttm DESC
            ) = 1
        ),
        historical_bridge AS (
            SELECT
                c.cohort_month,
                snap.account_uri,
                snap.organization_uri AS org_uri
            FROM (SELECT DISTINCT cohort_month FROM WORKSPACE.digitalda_stage.Account_ID_features_monthly) c
            JOIN PREP.MDM_RELTIO.f_entity_wxaccountnumber_organization_snapshot snap
                ON c.cohort_month >= '2026-03-01'::DATE
                AND c.cohort_month >= snap.row_eff_begin_dttm
                AND (c.cohort_month < snap.row_eff_end_dttm OR snap.row_eff_end_dttm IS NULL)
            UNION ALL
            SELECT 
                c.cohort_month,
                a.account_uri,
                a.org_uri
            FROM (SELECT DISTINCT cohort_month FROM WORKSPACE.digitalda_stage.Account_ID_features_monthly) c
            CROSS JOIN anchor_records a
            WHERE c.cohort_month < '2026-03-01'::DATE
        )
        SELECT * FROM historical_bridge
    ) hb ON wx.uri = hb.account_uri AND af.cohort_month = hb.cohort_month
),

ACCOUNT_CLOSED AS (
    SELECT 
        SOURCE_ACCOUNT_ID AS ACCOUNT_ID,
        MAX(CASE 
            WHEN ACCOUNT_CLOSED_DATE >= '3000-01-01'::DATE THEN NULL 
            ELSE ACCOUNT_CLOSED_DATE 
        END) AS CLOSED_DATE,
        MAX(ATTRITION_TYPE) AS ATTRITION_TYPE
    FROM FINANCE_ANALYTICS.NAM_PORTFOLIO_METRICS.NAM_ACCOUNT_ATTRITION
    GROUP BY SOURCE_ACCOUNT_ID
)

SELECT 
    af.account_id AS ACCOUNT_ID,
    af.cohort_month AS VOL_MONTH,
    af.gallons_mth AS GALLONS,
    af.outstanding_cardcount_mth AS OUTSTANDING_CARDS,
    af.active_cardcount_mth AS ACTIVE_CARDS,
    af.revenue_mth AS REVENUE,
    COALESCE(b.ORG_URI, 'UNMAPPED') AS ORG_URI,
    ac.CLOSED_DATE,
    ac.ATTRITION_TYPE,
    af.partner_ind AS PARTNER_IND
FROM WORKSPACE.digitalda_stage.Account_ID_features_monthly af
LEFT JOIN RELTIO_BRIDGE b ON af.account_id = b.ACCOUNT_ID AND af.cohort_month = b.COHORT_MONTH
LEFT JOIN ACCOUNT_CLOSED ac ON af.account_id = ac.ACCOUNT_ID
"""

print("Executing query and fetching pipeline data to pandas...")
snowpark_df = session.sql(query)
df = snowpark_df.to_pandas()
session.close()

# Clean dates
df['VOL_MONTH'] = pd.to_datetime(df['VOL_MONTH'])
df['CLOSED_DATE'] = pd.to_datetime(df['CLOSED_DATE'])

print(f"Loaded {len(df)} rows of history from pipeline tables.")

## 2. Define Auditing and Aggregation Lógica

We define the analysis logic with parameter toggles to test different filter definitions:

In [ ]:
def run_attrition_analysis(df, reporting_month_str, card_basis='outstanding_cards', exclude_partner=True, exclude_involuntary=False, exclude_conversions=False):
    M = pd.to_datetime(reporting_month_str)
    M_prev = M - pd.DateOffset(years=1)
    
    analysis_df = df.copy()
    if exclude_partner:
        # Exclude partner-billed accounts to align with MOR metrics
        analysis_df = analysis_df[analysis_df['PARTNER_IND'] == 'Wex']
        
    if exclude_involuntary:
        # Exclude involuntary attrition (fraud/credit closures)
        analysis_df = analysis_df[analysis_df['ATTRITION_TYPE'] != 'Involuntary']
        
    if exclude_conversions:
        # Exclude product conversions
        analysis_df = analysis_df[analysis_df['ATTRITION_TYPE'] != 'Conversion']
        
    # Active month is M_prev
    active_in_prev = analysis_df[analysis_df['VOL_MONTH'] == M_prev].copy()
    
    # Accounts that closed between M_prev and M
    closed_accounts = active_in_prev[
        (active_in_prev['CLOSED_DATE'] > M_prev) & 
        (active_in_prev['CLOSED_DATE'] <= M)
    ].copy()
    
    # Determine complete vs. partial churn
    active_in_M = analysis_df[
        (analysis_df['VOL_MONTH'] == M) & 
        ((analysis_df['CLOSED_DATE'].isna()) | (analysis_df['CLOSED_DATE'] > M))
    ]
    active_orgs_in_M = set(active_in_M['ORG_URI'].dropna().unique())
    
    # Complete = No accounts active in M for this ORG
    # Partial = At least one account active in M for this ORG
    closed_accounts['ATTRITION_CLASS'] = closed_accounts['ORG_URI'].apply(
        lambda org: 'At least ONE Account' if org in active_orgs_in_M else 'ALL Accounts Termi'
    )
    
    # Define Card Range grouping function
    card_col = 'OUTSTANDING_CARDS' if card_basis == 'outstanding_cards' else 'ACTIVE_CARDS'
    
    def get_range(cards): 
        if pd.isna(cards) or cards is None:
            return 'Null cards'
        elif cards == 1:
            return '1'
        elif cards == 2:
            return '2'
        elif 3 <= cards <= 4:
            return '3-4'
        elif 5 <= cards <= 9:
            return '5-9'
        elif 10 <= cards <= 25:
            return '10-25'
        elif 26 <= cards <= 50:
            return '26-50'
        elif 51 <= cards <= 250:
            return '51-250'
        elif 251 <= cards <= 1000:
            return '251-1000'
        elif cards > 1000:
            return '1000+'
        else:
            return 'Null cards'
            
    # Grouping A: Account-level card count in M_prev
    closed_accounts['ACCOUNT_CARD_RANGE'] = closed_accounts[card_col].apply(get_range)
    
    # Grouping B: Entity-level card count in M_prev
    entity_cards = active_in_prev.groupby('ORG_URI')[card_col].sum().reset_index()
    entity_cards['ENTITY_CARD_RANGE'] = entity_cards[card_col].apply(get_range)
    
    closed_accounts = closed_accounts.merge(
        entity_cards[['ORG_URI', 'ENTITY_CARD_RANGE']], 
        on='ORG_URI', 
        how='left'
    )
    closed_accounts['ENTITY_CARD_RANGE'] = closed_accounts['ENTITY_CARD_RANGE'].fillna('Null cards')
    
    return closed_accounts

## 3. Formatting Summary Tables

We create a helper function to aggregate and display the tables exactly like the Excel/Google Sheets tabs.

In [ ]:
def print_summary_tables(closed_df, range_col_name):
    ranges = ['1', '2', '3-4', '5-9', '10-25', '26-50', '51-250', '251-1000', '1000+', 'Null cards']
    
    summary = []
    for r in ranges:
        sub = closed_df[closed_df[range_col_name] == r]
        
        tot_rev = sub['REVENUE'].sum()
        all_termi_rev = sub[sub['ATTRITION_CLASS'] == 'ALL Accounts Termi']['REVENUE'].sum()
        one_active_rev = sub[sub['ATTRITION_CLASS'] == 'At least ONE Account']['REVENUE'].sum()
        pct_active_rev = (one_active_rev / tot_rev) * 100 if tot_rev > 0 else 0
        
        tot_gal = sub['GALLONS'].sum()
        all_termi_gal = sub[sub['ATTRITION_CLASS'] == 'ALL Accounts Termi']['GALLONS'].sum()
        one_active_gal = sub[sub['ATTRITION_CLASS'] == 'At least ONE Account']['GALLONS'].sum()
        pct_active_gal = (one_active_gal / tot_gal) * 100 if tot_gal > 0 else 0
        
        summary.append({
            'Card Range': r,
            'Revenue Attrition ($)': tot_rev,
            'ALL Accounts Termi ($)': all_termi_rev,
            'At least ONE Account ($)': one_active_rev,
            '% w/Active (Rev)': f"{pct_active_rev:.1f}%",
            'Gallon Attrition': tot_gal,
            'ALL Accounts Termi (Gal)': all_termi_gal,
            'At least ONE Account (Gal)': one_active_gal,
            '% w/Active (Gal)': f"{pct_active_gal:.1f}%"
        })
        
    summary_df = pd.DataFrame(summary)
    
    # Add Total Row
    total_rev = closed_df['REVENUE'].sum()
    total_all_termi_rev = closed_df[closed_df['ATTRITION_CLASS'] == 'ALL Accounts Termi']['REVENUE'].sum()
    total_one_active_rev = closed_df[closed_df['ATTRITION_CLASS'] == 'At least ONE Account']['REVENUE'].sum()
    total_pct_active_rev = (total_one_active_rev / total_rev) * 100 if total_rev > 0 else 0
    
    total_gal = closed_df['GALLONS'].sum()
    total_all_termi_gal = closed_df[closed_df['ATTRITION_CLASS'] == 'ALL Accounts Termi']['GALLONS'].sum()
    total_one_active_gal = closed_df[closed_df['ATTRITION_CLASS'] == 'At least ONE Account']['GALLONS'].sum()
    total_pct_active_gal = (total_one_active_gal / total_gal) * 100 if total_gal > 0 else 0
    
    total_row = {
        'Card Range': 'Total',
        'Revenue Attrition ($)': total_rev,
        'ALL Accounts Termi ($)': total_all_termi_rev,
        'At least ONE Account ($)': total_one_active_rev,
        '% w/Active (Rev)': f"{total_pct_active_rev:.1f}%",
        'Gallon Attrition': total_gal,
        'ALL Accounts Termi (Gal)': total_all_termi_gal,
        'At least ONE Account (Gal)': total_one_active_gal,
        '% w/Active (Gal)': f"{total_pct_active_gal:.1f}%"
    }
    
    summary_df = pd.concat([summary_df, pd.DataFrame([total_row])], ignore_index=True)
    
    # Format currency columns
    for col in ['Revenue Attrition ($)', 'ALL Accounts Termi ($)', 'At least ONE Account ($)']:
        summary_df[col] = summary_df[col].apply(lambda x: f"${x:,.2f}" if isinstance(x, (int, float)) else x)
    
    # Format integer columns
    for col in ['Gallon Attrition', 'ALL Accounts Termi (Gal)', 'At least ONE Account (Gal)']:
        summary_df[col] = summary_df[col].apply(lambda x: f"{x:,.0f}" if isinstance(x, (int, float)) else x)
        
    return summary_df

## 4. Run Audit for Feb 2026 (Compare with 'FEB 26 Reltio' Tab)

This runs the analysis for reporting month `2026-02-28`. We compare Option A (Account level cards) and Option B (Entity level cards).

In [ ]:
closed_feb26 = run_attrition_analysis(
    df, 
    '2026-02-28', 
    card_basis='outstanding_cards', 
    exclude_partner=True,
    exclude_involuntary=False,
    exclude_conversions=False
)

print("=== OPTION A: ACCOUNT-LEVEL CARD GROUPING ===")
display(print_summary_tables(closed_feb26, 'ACCOUNT_CARD_RANGE'))

print("\n=== OPTION B: ENTITY-LEVEL CARD GROUPING ===")
display(print_summary_tables(closed_feb26, 'ENTITY_CARD_RANGE'))

## 5. Segment-level Consolidation (Compare with 'APR 26' Segment Summary Tab)

This aggregates the results into the four segments: **Micro**, **Small**, **Medium**, and **Large**.

In [ ]:
def print_segment_summary(closed_df, range_col_name):
    def get_segment(r):
        if r in ['1', '2', '3-4']:
            return 'Micro'
        elif r in ['5-9', '10-25']:
            return 'Small'
        elif r in ['26-50', '51-250']:
            return 'Medium'
        elif r in ['251-1000', '1000+']:
            return 'Large'
        else:
            return 'Null/Other'
            
    closed_df['SEGMENT'] = closed_df[range_col_name].apply(get_segment)
    
    summary = closed_df.groupby('SEGMENT').agg(
        Revenue_Attrition=('REVENUE', 'sum'),
        Gallon_Attrition=('GALLONS', 'sum'),
        Account_Attrition_Count=('ACCOUNT_ID', 'count')
    ).reset_index()
    
    # Calculate totals
    total_row = pd.DataFrame([{
        'SEGMENT': 'Total',
        'Revenue_Attrition': summary['Revenue_Attrition'].sum(),
        'Gallon_Attrition': summary['Gallon_Attrition'].sum(),
        'Account_Attrition_Count': summary['Account_Attrition_Count'].sum()
    }])
    
    summary = pd.concat([summary, total_row], ignore_index=True)
    
    summary['Revenue_Attrition'] = summary['Revenue_Attrition'].apply(lambda x: f"${x:,.2f}")
    summary['Gallon_Attrition'] = summary['Gallon_Attrition'].apply(lambda x: f"{x:,.0f}")
    
    return summary

print("=== SEGMENT SUMMARY FOR FEB 2026 (ENTITY CARD BASIS) ===")
display(print_segment_summary(closed_feb26, 'ENTITY_CARD_RANGE'))

## 6. Audit for April 2026 (Compare with 'APR 26' Tab)

This runs the analysis for reporting month `2026-04-30` (using active month `2025-04-30`).

In [ ]:
closed_apr26 = run_attrition_analysis(df, '2026-04-30', card_basis='outstanding_cards', exclude_partner=True)

print("=== OPTION B: ENTITY-LEVEL CARD GROUPING FOR APR 2026 ===")
display(print_summary_tables(closed_apr26, 'ENTITY_CARD_RANGE'))

print("\n=== SEGMENT SUMMARY FOR APR 2026 (ENTITY CARD BASIS) ===")
display(print_segment_summary(closed_apr26, 'ENTITY_CARD_RANGE'))